# Personeelsplanning met SCIP (PySCIPOpt)

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/)

In dit notebook los je een **rosterprobleem** op: wie werkt wanneer welke dienst?
We modelleren het als een *gemengd-geheeltallig lineair programma* (MIP) en lossen het op met
[SCIP](https://www.scipopt.org/) via de Python-interface [PySCIPOpt](https://github.com/scipopt/PySCIPOpt).

**Wat leer je?**
1. Binaire beslissingsvariabelen definiëren
2. Harde randvoorwaarden (dekking, rusttijden, beschikbaarheid) formuleren
3. Een doelfunctie met zachte voorkeuren en eerlijkheid opstellen
4. De oplossing visualiseren en het model aanpassen (what-if)

Voer de cellen uit met **Shift + Enter**.

## 1. Probleembeschrijving

Een team van 8 medewerkers moet één week (maandag tot zondag) drie diensten bemannen:
**Vroeg**, **Laat** en **Nacht**.

| Regel | Type |
|---|---|
| Elke dienst heeft een vast aantal medewerkers nodig | hard |
| Maximaal één dienst per medewerker per dag | hard |
| Na een *Laat* of *Nacht* geen *Vroeg* de volgende dag; na *Nacht* ook geen *Laat* | hard |
| Per week minimaal 4 en maximaal 5 diensten | hard |
| Niet-beschikbare dagen (verlof, ziekte, ...) | hard |
| Maximaal 3 nachtdiensten per week; sommige medewerkers doen geen nachten | hard |
| Gevraagde vrije dagen respecteren | **zacht** (straf) |
| Nachtdiensten en weekenddiensten eerlijk verdelen | **zacht** (straf) |

## 2. Gegevens

In [ ]:
from pyscipopt import Model, quicksum
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

medewerkers = ["Ann", "Bram", "Chloë", "Dries", "Elise", "Finn", "Greet", "Hugo"]
dagen       = ["Ma", "Di", "Wo", "Do", "Vr", "Za", "Zo"]
diensten    = ["Vroeg", "Laat", "Nacht"]

# Benodigde bezetting per dienst per dag (Ma..Zo)
nodig = {
    "Vroeg": [2, 2, 2, 2, 2, 2, 2],
    "Laat":  [2, 2, 2, 2, 2, 1, 1],
    "Nacht": [1, 1, 1, 1, 1, 1, 1],
}

# Harde beperkingen: (medewerker, dag-index) is NIET beschikbaar
niet_beschikbaar = {("Bram", 2), ("Bram", 3), ("Greet", 5), ("Greet", 6), ("Finn", 0)}

# Zachte wensen: (medewerker, dag-index) wil graag vrij zijn
wil_vrij = {("Ann", 4), ("Chloë", 5), ("Chloë", 6), ("Hugo", 1), ("Elise", 0)}

# Medewerkers die geen nachtdiensten doen
geen_nacht = {"Elise"}

MIN_DIENSTEN, MAX_DIENSTEN, MAX_NACHTEN = 4, 5, 3

## 3. Het model

We gebruiken één binaire variabele per combinatie medewerker × dag × dienst:

$$x_{m,d,s} = \begin{cases} 1 & \text{als medewerker } m \text{ op dag } d \text{ dienst } s \text{ werkt} \\ 0 & \text{anders} \end{cases}$$

**Doelfunctie** (minimaliseren):

$$10 \cdot (\text{geschonden vrije wensen}) + 3 \cdot (\max\text{nachten} - \min\text{nachten}) + 2 \cdot (\max\text{weekend} - \min\text{weekend})$$

In [ ]:
def bouw_en_los_op(nodig=nodig, niet_beschikbaar=niet_beschikbaar, wil_vrij=wil_vrij,
                   geen_nacht=geen_nacht, tijdslimiet=60):
    M, D, S = range(len(medewerkers)), range(len(dagen)), range(len(diensten))
    V, L, N = 0, 1, 2   # indices van de diensten

    model = Model("personeelsplanning")
    model.hideOutput()

    x = {(m, d, s): model.addVar(vtype="B", name=f"x_{m}_{d}_{s}")
         for m in M for d in D for s in S}

    # 1. Dekking: exact het benodigde aantal per dienst
    for d in D:
        for s in S:
            model.addCons(quicksum(x[m, d, s] for m in M) == nodig[diensten[s]][d])

    # 2. Maximaal één dienst per dag
    for m in M:
        for d in D:
            model.addCons(quicksum(x[m, d, s] for s in S) <= 1)

    # 3. Rusttijden
    for m in M:
        for d in range(len(dagen) - 1):
            model.addCons(x[m, d, L] + x[m, d + 1, V] <= 1)   # laat -> vroeg
            model.addCons(x[m, d, N] + x[m, d + 1, V] <= 1)   # nacht -> vroeg
            model.addCons(x[m, d, N] + x[m, d + 1, L] <= 1)   # nacht -> laat

    # 4. Aantal diensten per week
    for m in M:
        totaal = quicksum(x[m, d, s] for d in D for s in S)
        model.addCons(totaal >= MIN_DIENSTEN)
        model.addCons(totaal <= MAX_DIENSTEN)

    # 5. Beschikbaarheid
    for m in M:
        for d in D:
            if (medewerkers[m], d) in niet_beschikbaar:
                for s in S:
                    model.addCons(x[m, d, s] == 0)

    # 6. Nachtdiensten: maximum en uitzonderingen
    nachten = {m: quicksum(x[m, d, N] for d in D) for m in M}
    for m in M:
        model.addCons(nachten[m] <= MAX_NACHTEN)
        if medewerkers[m] in geen_nacht:
            model.addCons(nachten[m] == 0)

    # 7. Eerlijkheid: spreiding van nacht- en weekenddiensten
    nmax = model.addVar(lb=0, name="nmax"); nmin = model.addVar(lb=0, name="nmin")
    wmax = model.addVar(lb=0, name="wmax"); wmin = model.addVar(lb=0, name="wmin")
    weekend = {m: quicksum(x[m, d, s] for d in (5, 6) for s in S) for m in M}
    for m in M:
        if medewerkers[m] not in geen_nacht:
            model.addCons(nachten[m] <= nmax)
            model.addCons(nachten[m] >= nmin)
        model.addCons(weekend[m] <= wmax)
        model.addCons(weekend[m] >= wmin)

    # Doelfunctie
    straf_wensen = quicksum(x[m, d, s] for m in M for d in D for s in S
                            if (medewerkers[m], d) in wil_vrij)
    model.setObjective(10 * straf_wensen + 3 * (nmax - nmin) + 2 * (wmax - wmin), "minimize")
    model.setParam("limits/time", tijdslimiet)
    model.optimize()
    return model, x


model, x = bouw_en_los_op()
print("Status:", model.getStatus())
if model.getNSols() > 0:
    print("Doelwaarde:", model.getObjVal(), "| gap:", round(model.getGap(), 4))

## 4. Resultaat

In [ ]:
def rooster(model, x):
    tabel = pd.DataFrame("-", index=medewerkers, columns=dagen)
    for (m, d, s), var in x.items():
        if model.getVal(var) > 0.5:
            tabel.iloc[m, d] = diensten[s]
    return tabel

if model.getNSols() == 0:
    print("Geen oplossing gevonden (probleem onhaalbaar?).")
else:
    tabel = rooster(model, x)
    display(tabel)

In [ ]:
def teken(tabel, wil_vrij=wil_vrij, niet_beschikbaar=niet_beschikbaar):
    codes = {"-": 0, "Vroeg": 1, "Laat": 2, "Nacht": 3}
    waarden = tabel.replace(codes).astype(int).values
    kleuren = ListedColormap(["#f2f2f2", "#ffd966", "#f4b183", "#8faadc"])

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.imshow(waarden, cmap=kleuren, vmin=0, vmax=3)
    ax.set_xticks(range(len(dagen)), dagen)
    ax.set_yticks(range(len(medewerkers)), medewerkers)
    ax.xaxis.tick_top()
    for i, m in enumerate(medewerkers):
        for j in range(len(dagen)):
            tekst = tabel.iloc[i, j][0] if tabel.iloc[i, j] != "-" else ""
            ax.text(j, i, tekst, ha="center", va="center", fontweight="bold")
            if (m, j) in niet_beschikbaar:
                ax.text(j, i, "✗", ha="center", va="center", color="red", alpha=.5, fontsize=16)
            if (m, j) in wil_vrij:
                ax.add_patch(plt.Rectangle((j-.5, i-.5), 1, 1, fill=False, ec="#2e9e4f", lw=3, zorder=5))
    ax.set_xticks([x-.5 for x in range(len(dagen)+1)], minor=True)
    ax.set_yticks([y-.5 for y in range(len(medewerkers)+1)], minor=True)
    ax.grid(which="minor", color="white", linewidth=2)
    ax.tick_params(which="both", length=0)
    ax.set_title("V = Vroeg · L = Laat · N = Nacht · ✗ = niet beschikbaar · groen kader = wens vrij", fontsize=9, pad=28)
    plt.tight_layout(); plt.show()

if model.getNSols() > 0:
    teken(tabel)

## 5. Controle en statistieken

In [ ]:
if model.getNSols() > 0:
    stats = pd.DataFrame({
        "Diensten": (tabel != "-").sum(axis=1),
        "Vroeg":    (tabel == "Vroeg").sum(axis=1),
        "Laat":     (tabel == "Laat").sum(axis=1),
        "Nacht":    (tabel == "Nacht").sum(axis=1),
        "Weekend":  (tabel[["Za", "Zo"]] != "-").sum(axis=1),
    })
    display(stats)

    geschonden = [(m, dagen[d]) for (m, d) in wil_vrij
                  if tabel.loc[m, dagen[d]] != "-"]
    print("Geschonden wensen:", geschonden if geschonden else "geen")

## 6. Opdrachten (what-if)

Pas de gegevens in sectie 2 aan, voer opnieuw uit en bekijk wat er gebeurt:

1. **Meer wensen**: voeg wensen toe aan `wil_vrij`, bv. `("Dries", 0)`, `("Dries", 1)`, `("Finn", 4)`. Blijf toevoegen tot de eerste wens níet meer gehonoreerd kan worden (zie sectie 5). Waarom precies die?
2. **Onhaalbaar maken**: zet `MAX_DIENSTEN = 4`. Het model is nu onhaalbaar (`infeasible`). Reken het na: totaal benodigde diensten (33) vs. capaciteit (`8 × MAX_DIENSTEN`).
3. **Extra regel**: voeg een regel toe zodat niemand meer dan **4 dagen na elkaar** werkt (tip: som over een venster van 5 opeenvolgende dagen ≤ 4).
4. **Ander gewicht**: wat gebeurt er als eerlijkheid zwaarder weegt (bijv. `10 * (nmax - nmin)`) dan de wensen?
5. **Uitbreiding**: laat medewerkers een *voorkeursdienst* opgeven en straf afwijkingen.

---
*Gemaakt voor gebruik op [mybinder.org](https://mybinder.org). Solver: [SCIP](https://www.scipopt.org/) (Apache 2.0).*